# 📊 Exploring Sign Language Datasets

This notebook walks through loading and exploring several popular sign language datasets.

**Datasets covered:**
- ASL-MNIST (CSV, 28×28 images)
- ASL Alphabet (directory of images)
- BdSL Sensor Glove (sensor time-series)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

## 1. ASL-MNIST (CSV Format)

Download from Kaggle:
```bash
kaggle datasets download -d datamunge/sign-language-mnist
unzip sign-language-mnist.zip -d data/asl_mnist/
```

In [ ]:
from scripts.data_loader import ASLMNISTDataset

# Adjust paths to where you extracted the data
train_csv = '../data/asl_mnist/sign_mnist_train.csv'
test_csv  = '../data/asl_mnist/sign_mnist_test.csv'

train_ds = ASLMNISTDataset(train_csv)
test_ds  = ASLMNISTDataset(test_csv)

print(f'Train samples: {len(train_ds)}')
print(f'Test samples:  {len(test_ds)}')
print(f'Classes: {train_ds.num_classes}')

In [ ]:
# Visualise a grid of samples
fig, axes = plt.subplots(3, 6, figsize=(14, 7))
axes = axes.ravel()

for i, ax in enumerate(axes):
    sample = train_ds[i * 300]  # spread across classes
    img = sample['image'].numpy().squeeze()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {sample["label"].item()}', fontsize=9)
    ax.axis('off')

plt.suptitle('ASL-MNIST Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution
labels = train_ds.labels
plt.figure(figsize=(10, 4))
sns.countplot(x=labels)
plt.title('ASL-MNIST Label Distribution')
plt.xlabel('Class Label')
plt.ylabel('Count')
plt.show()

print(f'Min per class: {min(np.bincount(labels))}')
print(f'Max per class: {max(np.bincount(labels))}')

## 2. ASL Alphabet (Image Directory)

Download from Kaggle:
```bash
kaggle datasets download -d grassknoted/asl-alphabet
unzip asl-alphabet.zip -d data/asl_alphabet/
```

Expected structure:
```
data/asl_alphabet/
├── train/
│   ├── A/
│   ├── B/
│   └── ...
└── test/
```

In [ ]:
from scripts.data_loader import ASLAlphabetDataset
from torchvision import transforms

# Basic transform for exploration
tfm = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Uncomment after downloading:
# asl_ds = ASLAlphabetDataset(root_dir='../data/asl_alphabet', split='train', transform=tfm)
# print(f'Samples: {len(asl_ds)}, Classes: {asl_ds.num_classes}')
# print(f'Class names: {asl_ds.class_names}')

print('(Uncomment above after downloading the ASL Alphabet dataset)')

## 3. BdSL Sensor Glove (Time-Series)

This dataset is included in the repo. No download needed.

Get it with:
```bash
python scripts/download_datasets.py --dataset bdsl-sensor-glove
```

In [ ]:
from scripts.data_loader import BdSLSensorGloveDataset

# Uncomment after downloading:
# sensor_ds = BdSLSensorGloveDataset(split='train')
# sample = sensor_ds[0]
# print(f'Sensor shape: {sample["sensors"].shape}')  # (max_len, 11)
# print(f'Gesture: {sample["gesture_id"]} ({sample["gesture_name"]})')
# print(f'Duration: {sample["duration_ms"]} ms')

print('(Uncomment after downloading the BdSL Sensor Glove dataset)')

## 4. Generic Image Directory Loader

For any dataset organised as `folder/class_name/images`:

In [ ]:
from scripts.data_loader import ImageDirectoryDataset

# Usage:
# ds = ImageDirectoryDataset(
#     root_dir='../data/my_dataset/train',
#     image_size=224,
#     augment=True,
# )
# print(f'{len(ds)} images across {ds.num_classes} classes')

print('(Provide your own image directory to use the generic loader)')

## Summary

| Loader | Format | Best For |
|--------|--------|----------|
| `ASLMNISTDataset` | CSV (pixels) | Quick prototyping, MNIST-style |
| `ASLAlphabetDataset` | Image dirs | CNN training on real hand images |
| `BdSLSensorGloveDataset` | JSON (sensors) | Temporal gesture recognition |
| `ImageDirectoryDataset` | Image dirs (generic) | Any custom image dataset |
| `GenericCSVDataset` | CSV (generic) | Any CSV with label+features |